# Claude Managed Agents로 Slack 데이터 분석 bot 만들기

## 소개

[`data_analyst_agent.ipynb`](data_analyst_agent.ipynb)의 agent를 Slack의 공식 앱 개발 framework인 [Bolt for Python](https://docs.slack.dev/tools/bolt-python/)으로 만든 Slack bot으로 감쌉니다. 질문과 CSV 첨부 파일을 넣어 bot을 mention하면 서술형 보고서가 thread로 돌아옵니다. 후속 message는 같은 세션을 이어갑니다.

```text
user: @databot Q1 매출을 이끄는 요인은 무엇인가요?  [sales.csv]
    │
    ▼
bot이 CSV를 업로드하고 agent 세션을 시작합니다
    │
    ▼
bot이 agent의 진행 상황을 thread로 streaming합니다
    │
    ▼
bot이 완성된 보고서를 thread에 게시합니다
```

### 배울 내용

- Slack mention으로 agent 실행 시작하기
- thread update로 agent의 진행 상황 보여 주기
- 완성된 보고서를 thread에 게시하기
- 후속 reply로 대화 이어가기

### 사전 요구 사항

1. 아래 설치 셀을 실행하세요.

2. [Slack app](https://api.slack.com/apps)을 생성하세요. **Create New App → From a manifest**를 선택하고 [`slack_app_manifest.yaml`](example_data/slack_data_bot/slack_app_manifest.yaml)을 붙여 넣은 뒤 workspace에 설치합니다. manifest는 Socket Mode(Slack이 WebSocket으로 event를 전달하므로 공개 URL이 필요 없음)와 필요한 scope를 활성화합니다. 그런 다음 token 두 개를 가져옵니다.

   - **OAuth & Permissions** → Bot User OAuth Token(`xoxb-...`) 복사
   - **Basic Information → App-Level Tokens** → `connections:write` scope로 생성(`xapp-...`)

   bot을 참여시키려는 channel에서 `/invite @databot`을 실행하세요.

3. [`data_analyst_agent.ipynb`](data_analyst_agent.ipynb)을 실행합니다. `ANALYST_ENV_ID`, `ANALYST_AGENT_ID`, `ANALYST_AGENT_VERSION`을 `.env`에 저장합니다.

아래 설정 셀은 Slack token을 입력받아 `.env`에 저장하므로 재시작할 때 다시 입력하지 않아도 됩니다(또는 미리 `.env`에 추가하면 입력을 건너뜁니다). `.env`는 이미 `.gitignore`에 있습니다. 절대로 version control에 commit하지 마세요. Slack workspace가 당장 없어도 코드를 읽어 볼 수 있습니다. 각 절은 무엇을 하는지 설명합니다. 하지만 bot을 실행하려면 workspace가 필요합니다.


In [ ]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv slack_bolt requests markdown-to-mrkdwn

In [ ]:
import io
import os
import threading
from getpass import getpass

import requests
from anthropic import Anthropic
from dotenv import load_dotenv, set_key
from markdown_to_mrkdwn import SlackMarkdownConverter
from slack_bolt import App
from slack_bolt.adapter.socket_mode import SocketModeHandler

load_dotenv(override=True)

# 첫 실행에 Slack token을 입력받아 .env에 저장합니다.
for key in ("SLACK_BOT_TOKEN", "SLACK_APP_TOKEN"):
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")
        set_key(".env", key, os.environ[key])

client = Anthropic()
app = App(token=os.environ["SLACK_BOT_TOKEN"])

for key in ("ANALYST_ENV_ID", "ANALYST_AGENT_ID", "ANALYST_AGENT_VERSION"):
    if not os.environ.get(key):
        raise RuntimeError(f"{key} not set. Run data_analyst_agent.ipynb first.")

# 데이터 분석 notebook이 저장한 ID로 설정합니다. agent와 environment를
# 재사용하면 bot을 재시작할 때마다 다시 마련하지 않아도 됩니다.
ANALYST_AGENT = {
    "id": os.environ["ANALYST_AGENT_ID"],
    "version": int(os.environ["ANALYST_AGENT_VERSION"]),
}
ANALYST_ENV_ID = os.environ["ANALYST_ENV_ID"]

# thread_ts -> session_id, 후속 message가 같은 세션에 도착하도록 합니다.
# 세션은 reply를 위해 열린 상태로 유지합니다. production에서는 이를 저장하고
# thread가 오래되면 세션을 archive하세요.
thread_sessions: dict[str, str] = {}

mrkdwn = SlackMarkdownConverter()

## 1. bot이 mention되면 세션 시작하기

Bolt는 모든 handler에 `ack` callback을 전달합니다. 이를 호출하면 Slack에 event를 받았음을 알립니다. Slack은 [3초 이내](https://docs.slack.dev/apis/events-api/#responding)에 확인되지 않은 모든 것을 재시도하므로 `on_mention`은 즉시 `ack()`를 호출하고 느린 작업(파일 업로드, 세션 생성, streaming)을 background thread의 `start_analysis`로 넘깁니다.

각 mention은 [Console](https://platform.claude.com/)의 **Sessions**에서 열어 전체 trace를 확인할 수 있는 세션을 생성합니다.


In [ ]:
@app.event("app_mention")
def on_mention(event, say, ack):
    ack()
    channel = event["channel"]
    thread_ts = event.get("thread_ts") or event["ts"]
    # mention text는 "<@BOTID> question"으로 도착합니다. mention prefix를 제거합니다.
    question = event["text"].split(">", 1)[-1].strip()
    slack_file = (event.get("files") or [None])[0]

    say(text="On it. Analyzing now.", thread_ts=thread_ts)
    # 이 handler가 Slack의 3초 제한 안에 반환되도록 느린 작업은
    # background thread에서 실행합니다.
    threading.Thread(target=start_analysis, args=(channel, thread_ts, question, slack_file)).start()


def start_analysis(channel: str, thread_ts: str, question: str, slack_file: dict | None) -> None:
    try:
        # mention에 파일이 첨부되었으면 Slack에서 가져와 세션이 mount할 수 있도록
        # Anthropic Files API에 다시 업로드합니다.
        resources = []
        if slack_file:
            resp = requests.get(
                slack_file["url_private"],
                headers={"Authorization": f"Bearer {app.client.token}"},
                timeout=30,
            )
            resp.raise_for_status()
            mime = slack_file.get("mimetype", "text/csv")
            uploaded = client.beta.files.upload(
                file=(slack_file["name"], io.BytesIO(resp.content), mime)
            )
            mount = "/mnt/session/uploads/data.csv"
            resources.append({"type": "file", "file_id": uploaded.id, "mount_path": mount})
            question += f"\n\nThe data is mounted at {mount}."

        # Slack thread당 세션 하나입니다. event stream을 읽는 사람이 reply 위치를
        # 알 수 있도록 metadata에 thread 좌표를 저장합니다.
        session = client.beta.sessions.create(
            environment_id=ANALYST_ENV_ID,
            agent={"type": "agent", **ANALYST_AGENT},
            resources=resources,
            # 제목은 최대 80자이며 Unicode control/format 문자를
            # 포함할 수 없습니다(Slack이 가끔 넣습니다).
            title="".join(c for c in question if c.isprintable())[:80],
            metadata={"slack_channel": channel, "slack_thread_ts": thread_ts},
        )
        thread_sessions[thread_ts] = session.id

        # 질문을 user.message event로 보냅니다. agent는 즉시 작업을 시작하며
        # relay_stream이 진행 상황을 thread에 게시합니다.
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": question}]}],
        )
        relay_stream(session.id, channel, thread_ts)
    except Exception as e:
        app.client.chat_postMessage(
            channel=channel, thread_ts=thread_ts, text=f"Analysis failed: {type(e).__name__}: {e}"
        )

## 2. 진행 상황과 결과를 thread로 전달하기

아래 정의한 `relay_stream` 함수는 두 API 사이를 연결합니다. Anthropic 세션 event stream에서 읽고 Slack에 게시합니다. agent가 idle이 될 때까지 반복한 뒤 최종 요약을 게시하고 agent가 작성한 파일을 업로드합니다.

`files.list(scope_id=...)`는 업로드한 CSV와 agent가 작성한 파일을 포함하여 세션의 모든 파일을 반환합니다. `downloadable == True`로 filtering하여 사용자의 입력 대신 agent가 생성한 출력(보고서, chart)만 Slack에 게시되게 합니다.


In [ ]:
def relay_stream(session_id: str, channel: str, thread_ts: str) -> None:
    summary = ""
    posted_progress = False
    for ev in client.beta.sessions.events.stream(session_id):
        t = ev.type
        if t == "agent.message":
            # 최신 text block을 유지합니다. 최종 요약이 됩니다.
            for b in ev.content:
                if b.type == "text" and b.text.strip():
                    summary = b.text
        elif t == "agent.tool_use" and not posted_progress:
            # agent가 명령을 실행하기 시작할 때 일회성
            # 진행 상황 update를 게시합니다.
            app.client.chat_postMessage(
                channel=channel, thread_ts=thread_ts, text="Running analysis..."
            )
            posted_progress = True
        elif t == "session.status_idle":
            break
        elif t == "session.status_terminated":
            trace = f"https://platform.claude.com/sessions/{session_id}"
            app.client.chat_postMessage(
                channel=channel,
                thread_ts=thread_ts,
                text=f"Session terminated unexpectedly. Trace: {trace}",
            )
            return

    # turn 완료. 요약을 게시하고 생성된 파일을 업로드합니다.
    if summary:
        text = mrkdwn.convert(summary)
        if len(text) > 3900:  # Slack text 제한 약 4000자
            text = text[:3900] + "\n_(truncated)_"
        app.client.chat_postMessage(channel=channel, thread_ts=thread_ts, text=text)
    outputs = client.beta.files.list(scope_id=session_id, betas=["managed-agents-2026-04-01"])
    for f in outputs.data:
        if not f.downloadable:
            continue
        content = client.beta.files.download(f.id).read()
        app.client.files_upload_v2(
            channel=channel, thread_ts=thread_ts, filename=f.filename, content=content
        )

## 3. 같은 세션에서 후속 message 처리하기

thread의 reply는 기존 세션의 다음 turn이 됩니다. bot을 다시 `@mention`할 필요가 없습니다. container filesystem과 대화 이력은 turn 사이에 유지됩니다.


In [ ]:
def continue_session(session_id: str, channel: str, thread_ts: str, text: str) -> None:
    try:
        client.beta.sessions.events.send(
            session_id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": text}]}],
        )
        relay_stream(session_id, channel, thread_ts)
    except Exception as e:
        app.client.chat_postMessage(
            channel=channel, thread_ts=thread_ts, text=f"Analysis failed: {type(e).__name__}: {e}"
        )


@app.event("message")
def on_thread_reply(event, ack):
    ack()
    thread_ts = event.get("thread_ts")
    # 이미 세션을 시작한 thread의 사람 reply만 처리합니다.
    # 편집/삭제와 다른 message subtype은 건너뜁니다.
    if event.get("subtype"):
        return
    if not thread_ts or event.get("bot_id") or thread_ts not in thread_sessions:
        return
    threading.Thread(
        target=continue_session,
        args=(thread_sessions[thread_ts], event["channel"], thread_ts, event["text"]),
    ).start()

## 4. bot 실행하기

아래 셀은 Slack에 연결하고 수신을 시작합니다. bot이 실행되는 동안 block됩니다. 완료하면 ■ interrupt 버튼으로 중단하세요.

bot이 있는 channel에서 CSV를 첨부하여 mention하세요. 진행 상황을 게시한 다음 thread에 요약과 `report.html`을 게시합니다.

<img src="https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/managed_agents/example_data/slack_data_bot/slack_thread.png" alt="bot의 분석을 보여 주는 Slack thread" width="600" />

같은 데이터를 더 깊이 살펴보려면 thread에서 reply하세요.


In [ ]:
SocketModeHandler(app, os.environ["SLACK_APP_TOKEN"]).start()

## 다음 단계

분석 agent를 Slack bot으로 감쌌습니다. mention이 세션을 시작하고, event stream이 진행 상황을 thread로 전달하며, 출력은 업로드되고, reply는 같은 대화를 이어갑니다.

- [`data_analyst_agent.ipynb`](data_analyst_agent.ipynb)의 agent system prompt를 바꿔 분석 스타일을 변경하세요. 해당 notebook을 다시 실행하면 새 agent를 생성하고 ID를 `.env`에 저장하여 bot이 사용하게 합니다.
- `thread_sessions`를 database에 저장하여 bot 재시작 후에도 대화가 유지되게 하세요.
- bot을 이 notebook 밖으로 옮기세요. 코드를 `.py` 파일로 복사하여 장시간 WebSocket 연결을 유지할 수 있는 곳에 deploy하세요.
